In [1]:
###########################################################
# Notebook 04 – Feature Engineering (Basis: 'price_totaal')
# Projekt: Preisbewertung NL
###########################################################

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor


In [3]:
#################################################
# 4.1. Vorbereitungen für Cross-Section Jahr 2023
#################################################

# Notwendige DataFrames laden
df_master = pd.read_csv("../data_clean/df_master.csv")
df_geo = pd.read_csv("../data_clean/df_geo.csv")

# Cross-Section für das Jahr 2023 erstellen
df_model = df_master[df_master['year'] == 2023].copy()

# Koordinaten aus df_geo ergänzen (falls noch nicht enthalten)
if 'longitude' not in df_model.columns:
    df_model = df_model.merge(
        df_geo[['municipality_code', 'longitude', 'latitude']],
        on='municipality_code',
        how='left'
    )

# Kontrolle
print(df_model.shape)
print()
df_model.head()


(1690, 50)



,municipality_code,year,price_total,price_egw,price_mgw,ratio_price_model,ratio_price_realized,share_egw,share_mgw,housing_stock_egw_x,...,population_dec31,migration_net,migration_per_1000,migration_rate,avg_space_egw,avg_space_mgw,province,ratio_egw_mgw,longitude,latitude
0,GM0014,2023,349931.0,417685.37108,306594.600316,1.362338,1.362338,0.390099,0.609901,47871.0,...,243768.0,5621.0,23.603069,2.360307,0.990298,0.985892,Groningen,1.362338,6.620667,53.218625
1,GM0014,2023,349931.0,417685.37108,306594.600316,1.362338,1.362338,0.390099,0.609901,47871.0,...,243768.0,5621.0,23.603069,2.360307,0.990298,0.985892,Groningen,1.362338,6.620667,53.218625
2,GM0014,2023,349931.0,417685.37108,306594.600316,1.362338,1.362338,0.390099,0.609901,47871.0,...,243768.0,5621.0,23.603069,2.360307,0.990298,0.985892,Groningen,1.362338,6.620667,53.218625
3,GM0014,2023,349931.0,417685.37108,306594.600316,1.362338,1.362338,0.390099,0.609901,47871.0,...,243768.0,5621.0,23.603069,2.360307,0.990298,0.985892,Groningen,1.362338,6.620667,53.218625
4,GM0014,2023,349931.0,417685.37108,306594.600316,1.362338,1.362338,0.390099,0.609901,47871.0,...,243768.0,5621.0,23.603069,2.360307,0.990298,0.985892,Groningen,1.362338,6.620667,53.218625


In [4]:
################################################
# 4.2. Zielvariable definieren (log_price_total)
################################################

import numpy as np

df_model['log_price_total'] = np.log(df_model['price_total'])


In [5]:
#################################################
# 4.3. Provinzen für One-Hot Encoding vorbereiten
#################################################

df_model['province_original'] = df_model['province']


In [6]:
#####################################
# 4.4. One-Hot Encoding der Provinzen
#####################################

df_model = pd.get_dummies(df_model, columns=['province'], drop_first=True)


In [7]:
df_model[['gemeentenaam'] + [col for col in df_model.columns if col.startswith('province_')]].head(10)


,gemeentenaam,province_original,province_Flevoland,province_Fryslân,province_Gelderland,province_Groningen,province_Limburg,province_Noord-Brabant,province_Noord-Holland,province_Overijssel,province_Utrecht,province_Zeeland,province_Zuid-Holland
0,Groningen (gemeente),Groningen,False,False,False,True,False,False,False,False,False,False,False
1,Groningen (gemeente),Groningen,False,False,False,True,False,False,False,False,False,False,False
2,Groningen (gemeente),Groningen,False,False,False,True,False,False,False,False,False,False,False
3,Groningen (gemeente),Groningen,False,False,False,True,False,False,False,False,False,False,False
4,Groningen (gemeente),Groningen,False,False,False,True,False,False,False,False,False,False,False
5,Almere,Flevoland,True,False,False,False,False,False,False,False,False,False,False
6,Almere,Flevoland,True,False,False,False,False,False,False,False,False,False,False
7,Almere,Flevoland,True,False,False,False,False,False,False,False,False,False,False
8,Almere,Flevoland,True,False,False,False,False,False,False,False,False,False,False
9,Almere,Flevoland,True,False,False,False,False,False,False,False,False,False,False


In [8]:
df_model[['gemeentenaam'] + [col for col in df_model.columns if col.startswith('province_')]].head(10)


,gemeentenaam,province_original,province_Flevoland,province_Fryslân,province_Gelderland,province_Groningen,province_Limburg,province_Noord-Brabant,province_Noord-Holland,province_Overijssel,province_Utrecht,province_Zeeland,province_Zuid-Holland
0,Groningen (gemeente),Groningen,False,False,False,True,False,False,False,False,False,False,False
1,Groningen (gemeente),Groningen,False,False,False,True,False,False,False,False,False,False,False
2,Groningen (gemeente),Groningen,False,False,False,True,False,False,False,False,False,False,False
3,Groningen (gemeente),Groningen,False,False,False,True,False,False,False,False,False,False,False
4,Groningen (gemeente),Groningen,False,False,False,True,False,False,False,False,False,False,False
5,Almere,Flevoland,True,False,False,False,False,False,False,False,False,False,False
6,Almere,Flevoland,True,False,False,False,False,False,False,False,False,False,False
7,Almere,Flevoland,True,False,False,False,False,False,False,False,False,False,False
8,Almere,Flevoland,True,False,False,False,False,False,False,False,False,False,False
9,Almere,Flevoland,True,False,False,False,False,False,False,False,False,False,False


In [9]:
###########################################
# 4.5. Lagevariable „log_density“ erstellen
###########################################

df_model['log_density'] = np.log(df_model['bevolkingsdichtheid'] + 1)


In [10]:
#################################
# 4.6. 'Randstad'-Dummy erstellen
#################################

randstad_provinces = ['Noord-Holland', 'Zuid-Holland', 'Utrecht', 'Flevoland']

df_model['is_randstad'] = df_model['province_original'].isin(randstad_provinces).astype(int)


In [11]:
province_dummies = [col for col in df_model.columns if col.startswith('province_')]

df_model = df_model.drop(columns=['province_original'], errors='ignore')


In [12]:
#############################
# 4.7. Skalierung durchführen
#############################

from sklearn.preprocessing import StandardScaler

# Liste der numerischen Variablen, die wir skalieren wollen
num_features = [
    'income_mean',
    'woz_mean',
    'ratio_price_model',
    'ratio_stock',
    'woningvoorraad_total',
    'pipeline_total',
    'new_build_stock_tot',
    'households',
    'log_density',
    'migration_per_1000',
    'employment_rate',
    'labor_participation_net'
]

# Skalierer initialisieren
scaler = StandardScaler()

# Skalierung anwenden
df_model[num_features] = scaler.fit_transform(df_model[num_features])


In [13]:
print(df_model[num_features].head())
print()
df_model[num_features].describe()


   income_mean  woz_mean  ratio_price_model  ratio_stock  \
0    -1.461978 -0.862786          -1.407024    -1.513529   
1    -1.461978 -0.862786          -1.407024    -1.513529   
2    -1.461978 -0.862786          -1.407024    -1.513529   
3    -1.461978 -0.862786          -1.407024    -1.513529   
4    -1.461978 -0.862786          -1.407024    -1.513529   

   woningvoorraad_total  pipeline_total  new_build_stock_tot  households  \
0              2.602552        1.298874             2.936873    2.939009   
1              2.602552        1.298874             2.936873    2.939009   
2              2.602552        1.298874             2.936873    2.939009   
3              2.602552        1.298874             2.936873    2.939009   
4              2.602552        1.298874             2.936873    2.939009   

   log_density  migration_per_1000  employment_rate  labor_participation_net  
0     0.844477            2.495057        -3.641399                -0.815213  
1     0.844477          

,income_mean,woz_mean,ratio_price_model,ratio_stock,woningvoorraad_total,pipeline_total,new_build_stock_tot,households,log_density,migration_per_1000,employment_rate,labor_participation_net
count,1.690000e+03,1.690000e+03,1.690000e+03,1.690000e+03,1.690000e+03,1.690000e+03,1.690000e+03,1.690000e+03,1.690000e+03,1.690000e+03,1.690000e+03,1.690000e+03
mean,1.765846e-16,-3.363516e-17,1.009055e-16,-2.018110e-16,5.045274e-17,1.681758e-17,-3.363516e-17,3.363516e-17,-3.195340e-16,5.255494e-17,-1.333634e-14,-1.194048e-15
std,1.000296e+00,1.000296e+00,1.000296e+00,1.000296e+00,1.000296e+00,1.000296e+00,1.000296e+00,1.000296e+00,1.000296e+00,1.000296e+00,1.000296e+00,1.000296e+00
min,-1.461978e+00,-1.941905e+00,-1.737879e+00,-1.715504e+00,-6.062924e-01,-3.365306e-01,-4.819150e-01,-5.796415e-01,-2.992158e+00,-4.162187e+00,-4.722559e+00,-5.134402e+00
25%,-4.979460e-01,-6.556822e-01,-8.043152e-01,-8.123469e-01,-3.671440e-01,-2.667586e-01,-3.244556e-01,-3.577796e-01,-7.363222e-01,-5.930538e-01,-3.979181e-01,-5.568003e-01
50%,-1.207161e-01,-1.106725e-01,-1.357095e-01,-1.389169e-01,-2.414711e-01,-2.067676e-01,-2.319978e-01,-2.440012e-01,-1.046698e-01,-6.822032e-02,2.507780e-01,1.446064e-01
75%,2.947834e-01,4.670379e-01,6.373226e-01,6.715489e-01,-9.394727e-03,-6.200695e-02,5.679923e-03,-3.243866e-02,7.663886e-01,3.781300e-01,6.832421e-01,6.614324e-01
max,1.118525e+01,6.015237e+00,3.563460e+00,3.611770e+00,1.185680e+01,1.389958e+01,1.425476e+01,1.189103e+01,2.454399e+00,4.991726e+00,1.548170e+00,2.913317e+00


In [14]:
##############################
# 4.8. Feature‑Satz definieren
##############################

# Zielvariable festlegen

target = 'log_price_total'

# Numerische Features
num_features = [
    'income_mean',
    'woz_mean',
    'log_density',
    'migration_per_1000',
    'employment_rate',
    'labor_participation_net'
]

# Lagevariablen
location_features = [
    'longitude',
    'latitude',
    'is_randstad'
]

# Sonstige Variablen
additional_features = [
    'ratio_price_model',
    'ratio_stock',
    'woningvoorraad_total',
    'pipeline_total',
    'new_build_stock_tot',
    'households'
]

# Provinz-Dummies
province_dummies = [col for col in df_model.columns if col.startswith('province_')]

# Finaler Feature-Satz
feature_cols = num_features + location_features + province_dummies + additional_features
len(feature_cols), feature_cols


(26,
 ['income_mean',
  'woz_mean',
  'log_density',
  'migration_per_1000',
  'employment_rate',
  'labor_participation_net',
  'longitude',
  'latitude',
  'is_randstad',
  'province_Flevoland',
  'province_Fryslân',
  'province_Gelderland',
  'province_Groningen',
  'province_Limburg',
  'province_Noord-Brabant',
  'province_Noord-Holland',
  'province_Overijssel',
  'province_Utrecht',
  'province_Zeeland',
  'province_Zuid-Holland',
  'ratio_price_model',
  'ratio_stock',
  'woningvoorraad_total',
  'pipeline_total',
  'new_build_stock_tot',
  'households'])

In [15]:
# Dataframe Modell-Dataframe erstellen

X = df_model[feature_cols]
y = df_model[target]


In [16]:
df_model.head(20)

,municipality_code,year,price_total,price_egw,price_mgw,ratio_price_model,ratio_price_realized,share_egw,share_mgw,housing_stock_egw_x,...,province_Groningen,province_Limburg,province_Noord-Brabant,province_Noord-Holland,province_Overijssel,province_Utrecht,province_Zeeland,province_Zuid-Holland,log_density,is_randstad
0,GM0014,2023,349931.0,417685.371080,306594.600316,-1.407024,1.362338,0.390099,0.609901,47871.0,...,True,False,False,False,False,False,False,False,0.844477,0
1,GM0014,2023,349931.0,417685.371080,306594.600316,-1.407024,1.362338,0.390099,0.609901,47871.0,...,True,False,False,False,False,False,False,False,0.844477,0
2,GM0014,2023,349931.0,417685.371080,306594.600316,-1.407024,1.362338,0.390099,0.609901,47871.0,...,True,False,False,False,False,False,False,False,0.844477,0
3,GM0014,2023,349931.0,417685.371080,306594.600316,-1.407024,1.362338,0.390099,0.609901,47871.0,...,True,False,False,False,False,False,False,False,0.844477,0
4,GM0014,2023,349931.0,417685.371080,306594.600316,-1.407024,1.362338,0.390099,0.609901,47871.0,...,True,False,False,False,False,False,False,False,0.844477,0
5,GM0034,2023,424531.0,495255.358010,251950.903571,-0.791282,1.965682,0.709317,0.290683,64328.0,...,False,False,False,False,False,False,False,False,1.128864,1
6,GM0034,2023,424531.0,495255.358010,251950.903571,-0.791282,1.965682,0.709317,0.290683,64328.0,...,False,False,False,False,False,False,False,False,1.128864,1
7,GM0034,2023,424531.0,495255.358010,251950.903571,-0.791282,1.965682,0.709317,0.290683,64328.0,...,False,False,False,False,False,False,False,False,1.128864,1
8,GM0034,2023,424531.0,495255.358010,251950.903571,-0.791282,1.965682,0.709317,0.290683,64328.0,...,False,False,False,False,False,False,False,False,1.128864,1
9,GM0034,2023,424531.0,495255.358010,251950.903571,-0.791282,1.965682,0.709317,0.290683,64328.0,...,False,False,False,False,False,False,False,False,1.128864,1


In [17]:
df_model[['gemeentenaam'] + [col for col in df_model.columns if col.startswith('province_')]].head(10)


,gemeentenaam,province_Flevoland,province_Fryslân,province_Gelderland,province_Groningen,province_Limburg,province_Noord-Brabant,province_Noord-Holland,province_Overijssel,province_Utrecht,province_Zeeland,province_Zuid-Holland
0,Groningen (gemeente),False,False,False,True,False,False,False,False,False,False,False
1,Groningen (gemeente),False,False,False,True,False,False,False,False,False,False,False
2,Groningen (gemeente),False,False,False,True,False,False,False,False,False,False,False
3,Groningen (gemeente),False,False,False,True,False,False,False,False,False,False,False
4,Groningen (gemeente),False,False,False,True,False,False,False,False,False,False,False
5,Almere,True,False,False,False,False,False,False,False,False,False,False
6,Almere,True,False,False,False,False,False,False,False,False,False,False
7,Almere,True,False,False,False,False,False,False,False,False,False,False
8,Almere,True,False,False,False,False,False,False,False,False,False,False
9,Almere,True,False,False,False,False,False,False,False,False,False,False


In [18]:
# Dataframes als csv-Datei speichern

df_model.to_csv('../data_clean/df_model_prepared.csv', index=False)
X.to_csv('../data_clean/X_features.csv', index=False)
y.to_csv('../data_clean/y_target.csv', index=False)


In [19]:
########
# Exkurs
########

df_model.columns

Index(['municipality_code', 'year', 'price_total', 'price_egw', 'price_mgw',
       'ratio_price_model', 'ratio_price_realized', 'share_egw', 'share_mgw',
       'housing_stock_egw_x', 'housing_stock_mgw_x', 'ratio_area',
       'ratio_stock', 'gemeentenaam', 'population', 'bevolkingsdichtheid',
       'households', 'woningvoorraad_total', 'housing_stock_egw_y',
       'housing_stock_mgw_y', 'living_space_total', 'living_space_egw',
       'living_space_mgw', 'income_mean', 'income_median', 'woz_mean',
       'pipeline_total', 'pipeline_started', 'pipeline_permit',
       'new_build_stock_tot', 'new_build_liv_space_tot', 'new_build_stock_egw',
       'new_build_liv_space_egw', 'new_build_stock_mgw',
       'new_build_liv_space_mgw', 'unemployment_rate',
       'labor_participation_gross', 'labor_participation_net',
       'employment_rate', 'population_jan1', 'population_dec31',
       'migration_net', 'migration_per_1000', 'migration_rate',
       'avg_space_egw', 'avg_space_mgw', 'ra

In [20]:
province_cols = [col for col in df_model.columns if col.startswith('province_')]

df_model[province_cols].sum().sort_values(ascending=False)


province_Noord-Brabant    274
province_Gelderland       250
province_Zuid-Holland     246
province_Noord-Holland    217
province_Limburg          155
province_Utrecht          130
province_Overijssel       125
province_Fryslân           90
province_Zeeland           65
province_Groningen         48
province_Flevoland         30
dtype: int64

In [21]:
df_model[province_cols].sum(axis=1).value_counts()


1    1630
0      60
Name: count, dtype: int64